# Cancelling a Task Execution — Stopping an In-Flight A2A Task

This notebook cancels a real, genuinely in-flight A2A task via
`cancel()`: dispatch a long Hailstone sequence, cancel it mid-step, and
confirm both that the task settles `TASK_STATE_CANCELED` and that the
underlying work actually stopped, not just that its status changed.

**NOTE**
There is no client-side cancel in this framework yet: `a2a/client/`
has no `cancel` at all, so `UseA2AAgentTool` cannot cancel a peer. This
notebook drives the raw `a2a-sdk` client directly, the same approach
`more-examples/ch10/streaming_llmagent_executor.ipynb` takes for
streaming.

`cancel()`'s ordering matters here: it has to publish `CANCELED` before
anything that can suspend, or a live subscriber can miss it entirely
(a real, previously confirmed failure mode, not a hypothetical one;
see `LLMAgentA2AExecutor.cancel()`'s own docstring). One open, known
gap this notebook doesn't exercise: if the agent completes normally at
the same instant a cancel arrives, both `CANCELED` and `COMPLETED` can
end up published for the same task, a separate issue already tracked
elsewhere, not something fixed here.

## Setup Instructions

This notebook launches the `extra/a2a-from-scratch-hailstone` and
`extra/a2a-from-scratch-hailstone-streaming` apps from a local checkout,
so it needs the full repository, not just the package. Launch it with:

```sh
uv run --with jupyter jupyter lab
```

from the project root. No PyPI release of `llm-agents-from-scratch` has
A2A support yet (still under "Unreleased" in `CHANGELOG.md`), so the
`pip install` cell below won't provide what this notebook needs even
if uncommented; it's left only for parity with the other notebooks.

In [1]:
# No PyPI release has A2A support yet -- see Setup Instructions above.
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have
Ollama installed on your local machine and have its LLM hosting
service running. To download Ollama, follow the instructions found on
this page: https://ollama.com/download. After downloading and
installing Ollama, you can start a service by opening a terminal and
running `ollama serve`.

In [2]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"\u2713 Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"\u2713 Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


use_cloud = "OLLAMA_API_KEY" in os.environ
ensure_ollama() if not use_cloud else print("\u2713 Using Ollama Cloud")

✓ Using Ollama Cloud


In [3]:
model = "glm-5.3:cloud" if use_cloud else "qwen3:14b"

## The Two Servers

The same standalone apps `more-examples/ch10/streaming_llmagent_executor.ipynb`
uses: `extra/a2a-from-scratch-hailstone` and its streaming sibling,
genuinely separate OS processes with their own log streams. 27 is the
starting number: its Hailstone sequence takes over 100 steps, which
leaves plenty of margin to cancel well before it would ever finish on
its own.

In [4]:
import contextlib
import signal
from pathlib import Path
from urllib.parse import urlparse


def find_repo_root(marker="extra"):
    """Walk up from cwd to the first ancestor containing `marker`.

    Jupyter sets a kernel's cwd to the directory of the opened notebook
    file under normal use, but that's not guaranteed for every launch
    method (e.g. running via nbconvert from the project root), so this
    doesn't assume a fixed number of parent hops.
    """
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / marker).is_dir():
            return candidate
    msg = f"Could not find a parent directory containing {marker!r}."
    raise RuntimeError(msg)


def ensure_hailstone_server(app_dir, host, log_path, timeout=15):
    """Start a from-scratch Hailstone A2A server app if not already running.

    Returns the Popen handle if this call started the server, or None if
    it was already running (so Cleanup knows not to tear it down). Runs
    in its own session (start_new_session=True) so Cleanup can kill the
    whole process group -- terminating only the `uv run` wrapper process
    can leave the actual uvicorn process orphaned and still bound to the
    port.
    """

    def _up():
        try:
            with urllib.request.urlopen(
                f"{host}/.well-known/agent-card.json",
                timeout=1,
            ):
                pass
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        print(f"\u2713 {app_dir} A2A server already running at {host}")
        return None

    server_path = find_repo_root() / "extra" / app_dir
    port = urlparse(host).port or 80
    env = dict(os.environ, OLLAMA_MODEL=model)
    if use_cloud:
        env["OLLAMA_HOST"] = "https://ollama.com"
    print(f"Starting {app_dir} A2A server at {host}...")
    with open(log_path, "w") as log_file:
        process = subprocess.Popen(
            [
                "uv",
                "run",
                "uvicorn",
                "main:app",
                "--host",
                "0.0.0.0",
                "--port",
                str(port),
            ],
            cwd=server_path,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            env=env,
        )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            print(f"\u2713 {app_dir} A2A server up at {host}")
            return process
        time.sleep(0.5)

    stop_hailstone_server(process)
    raise RuntimeError(f"A2A server did not start within {timeout}s")


def stop_hailstone_server(process):
    """Kills the whole process group started by ensure_hailstone_server()."""

    def _signal(sig):
        with contextlib.suppress(ProcessLookupError):
            os.killpg(os.getpgid(process.pid), sig)

    _signal(signal.SIGTERM)
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        _signal(signal.SIGKILL)
        process.wait(timeout=5)


def cleanup_servers():
    """Stops both servers this notebook started, ignoring already-up ones.

    Looked up by name at call time rather than passed in, so later
    cells (discovery, dispatch) can call this from an except block on
    the same two module-level names Cleanup uses, without threading
    the process handles through every intervening cell.
    """
    for process in (streaming_process, nonstreaming_process):
        if process is not None:
            stop_hailstone_server(process)


started_servers = []
try:
    streaming_process = ensure_hailstone_server(
        "a2a-from-scratch-hailstone-streaming",
        "http://127.0.0.1:9301",
        "streaming_hailstone_server.log",
    )
    started_servers.append(streaming_process)
    nonstreaming_process = ensure_hailstone_server(
        "a2a-from-scratch-hailstone",
        "http://127.0.0.1:9300",
        "nonstreaming_hailstone_server.log",
    )
    started_servers.append(nonstreaming_process)
except Exception:
    for started in started_servers:
        if started is not None:
            stop_hailstone_server(started)
    raise

Starting a2a-from-scratch-hailstone-streaming A2A server at http://127.0.0.1:9301...


✓ a2a-from-scratch-hailstone-streaming A2A server up at http://127.0.0.1:9301
Starting a2a-from-scratch-hailstone A2A server at http://127.0.0.1:9300...


✓ a2a-from-scratch-hailstone A2A server up at http://127.0.0.1:9300


## Cancelling the Non-Streaming Peer

`LLMAgentA2AExecutor` publishes only a single terminal status, so
there's no incremental chunk to read a task id off while dispatch is
still in flight the way the streaming peer allows. Instead, a second,
completely independent client polls `list_tasks()`, which the SDK
backs with the same task store `execute()` persists to immediately on
submission, well before the agent's first LLM call returns.

Dispatch and cancel use two separate clients throughout this notebook,
on purpose: a real canceller (an operator, a monitoring service) is
never the same connection that dispatched the task in the first
place.

In [5]:
import asyncio
from contextlib import asynccontextmanager

import httpx
from a2a.client import ClientConfig, create_client
from a2a.helpers import new_text_message
from a2a.types import (
    CancelTaskRequest,
    ListTasksRequest,
    SendMessageRequest,
    TaskState,
)
from a2a.types import (
    Role as A2ARole,
)

from llm_agents_from_scratch.a2a import A2AAgentSpec

nonstreaming_spec = await A2AAgentSpec.from_url(
    "http://127.0.0.1:9300",
    timeout=120.0,
)

task_text = (
    "Compute the full hailstone sequence starting at 27, until it reaches 1."
)


@asynccontextmanager
async def open_client(agent_card, *, streaming):
    """A fresh SDK client on its own httpx connection every time.

    Never reused across dispatch and cancel: a real canceller has no
    reason to share a connection with whoever dispatched the task.
    """
    async with httpx.AsyncClient(timeout=120.0) as httpx_client:
        client = await create_client(
            agent=agent_card,
            client_config=ClientConfig(
                streaming=streaming,
                httpx_client=httpx_client,
            ),
        )
        try:
            yield client
        finally:
            await client.close()


async def dispatch_nonstreaming():
    """Send the task and return only once it settles. Never cancels it."""
    async with open_client(
        nonstreaming_spec.agent_card,
        streaming=False,
    ) as client:
        message = new_text_message(text=task_text, role=A2ARole.ROLE_USER)
        request = SendMessageRequest(message=message)
        last = None
        async for chunk in client.send_message(request):
            last = chunk
        return last


send_future = asyncio.ensure_future(dispatch_nonstreaming())

task_id = None
async with open_client(
    nonstreaming_spec.agent_card,
    streaming=False,
) as observer:
    for _ in range(50):
        await asyncio.sleep(0.2)
        listing = await observer.list_tasks(ListTasksRequest())
        for task in listing.tasks:
            state = TaskState.Name(task.status.state)
            if state in ("TASK_STATE_SUBMITTED", "TASK_STATE_WORKING"):
                task_id = task.id
        if task_id:
            break
    if task_id is None:
        send_future.cancel()
        msg = "Task never appeared in list_tasks()."
        raise RuntimeError(msg)
    print(f"Discovered in-flight task {task_id} via list_tasks()")

    cancel_result = await observer.cancel_task(CancelTaskRequest(id=task_id))
    print(
        "cancel_task() response state:",
        TaskState.Name(cancel_result.status.state),
    )

final_chunk = await send_future
final_state = TaskState.Name(final_chunk.task.status.state)
print("send_message()'s own final chunk state:", final_state)

/home/nerdai/Projects/llm-agents-from-scratch/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Discovered in-flight task a1b1391a-98d2-46af-ac96-4f32186c9dc1 via list_tasks()

cancel_task() response state:

TASK_STATE_CANCELED

send_message()'s own final chunk state:

TASK_STATE_CANCELED

`nonstreaming_hailstone_server.log` shows why this is a genuine
cancellation, not just a status flip: the log goes quiet right after
the first "Processing Step" line, with no further tool call ever
logged. The agent's own LLM call was still in flight when `cancel()`
tore down the background task underneath it.

In [6]:
with open("nonstreaming_hailstone_server.log") as f:
    print(f.read())

warning

:

`VIRTUAL_ENV=/home/nerdai/.cache/uv/builds-v0/.tmpe2fRLv` does not match the project environment path `.venv` and will be ignored; use `--active` to target the active environment instead


INFO:     Started server process [2293703]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9300 (Press CTRL+C to quit)
INFO:     127.0.0.1:51030 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:51042 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO (llm_agents_fs.LLMAgent) :      🚀 Starting task: Compute the full hailstone sequence starting at 27, until it reaches 1.
INFO (llm_agents_fs.TaskHandler) :      ⚙️ Processing Step: Compute the full hailstone sequence starting at 27, until it reaches 1.
INFO:     127.0.0.1:51058 - "POST / HTTP/1.1" 200 OK
INFO:     127.0.0.1:51058 - "POST / HTTP/1.1" 200 OK
INFO:     127.0.0.1:51048 - "POST / HTTP/1.1" 200 OK


## Cancelling the Streaming Peer

`StreamingLLMAgentA2AExecutor` publishes a `TaskStatusUpdateEvent`
naming the task as soon as it's created, so the task id is available
straight from the first streamed chunk, no polling needed. The actual
cancel call still goes through a separate, freshly opened client, same
as the non-streaming example above: streaming just changes how the
task id is *found*, not who's allowed to cancel it.

In [7]:
streaming_spec = await A2AAgentSpec.from_url(
    "http://127.0.0.1:9301",
    timeout=120.0,
)

task_id_holder = {}


async def dispatch_streaming():
    """Send the task and return only once it settles. Never cancels it."""
    async with open_client(streaming_spec.agent_card, streaming=True) as client:
        message = new_text_message(text=task_text, role=A2ARole.ROLE_USER)
        request = SendMessageRequest(message=message)
        chunks = []
        async for chunk in client.send_message(request):
            is_task = chunk.WhichOneof("payload") == "task"
            if is_task and not task_id_holder:
                task_id_holder["id"] = chunk.task.id
            chunks.append(chunk)
        return chunks


send_future = asyncio.ensure_future(dispatch_streaming())

for _ in range(50):
    await asyncio.sleep(0.1)
    if task_id_holder:
        break
task_id = task_id_holder.get("id")
print(f"Discovered task {task_id} from the first streamed chunk")

async with open_client(streaming_spec.agent_card, streaming=True) as observer:
    cancel_result = await observer.cancel_task(CancelTaskRequest(id=task_id))
    print(
        "cancel_task() response state:",
        TaskState.Name(cancel_result.status.state),
    )

chunks = await send_future
last_state = TaskState.Name(chunks[-1].status_update.status.state)
print(f"send_message() yielded {len(chunks)} chunks total")
print("last streamed chunk state:", last_state)

Discovered task 16084b0a-20a3-4189-9343-cedd5575f0b2 from the first streamed chunk

cancel_task() response state:

TASK_STATE_CANCELED

send_message() yielded 5 chunks total

last streamed chunk state:

TASK_STATE_CANCELED

Same story in `streaming_hailstone_server.log`: work stops right
after the first step begins, before any tool call completes.

In [8]:
with open("streaming_hailstone_server.log") as f:
    print(f.read())

warning

:

`VIRTUAL_ENV=/home/nerdai/.cache/uv/builds-v0/.tmpe2fRLv` does not match the project environment path `.venv` and will be ignored; use `--active` to target the active environment instead


INFO:     Started server process [2293687]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9301 (Press CTRL+C to quit)
INFO:     127.0.0.1:52016 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:33198 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO (llm_agents_fs.SupervisedTaskHandler) :      ⚙️ Processing Step: Compute the full hailstone sequence starting at 27, until it reaches 1.
INFO:     127.0.0.1:33200 - "POST / HTTP/1.1" 200 OK
INFO:     127.0.0.1:33204 - "POST / HTTP/1.1" 200 OK


## Cleanup

In [9]:
cleanup_servers()